# Paper Scraper

We scrap papers from the main conference track of the 2024 NeurIPS conference

In [1]:
pip install requests beautifulsoup4 pymupdf4llm tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pymupdf4llm
from pathlib import Path
from tqdm import tqdm
import signal
from contextlib import contextmanager

# Configuration
BASE_URL = "https://proceedings.neurips.cc"
PAPERS_URL = f"{BASE_URL}/paper_files/paper/2024"
OUTPUT_DIR = "neurips_2024_papers"
MARKDOWN_DIR = os.path.join(OUTPUT_DIR, "markdown")
PDF_DIR = os.path.join(OUTPUT_DIR, "pdfs")
SKIPPED_FILE = os.path.join(OUTPUT_DIR, "skipped_papers.txt")
PDF_TIMEOUT = 180  # 3 minutes timeout for PDF conversion

# Create output directories
os.makedirs(MARKDOWN_DIR, exist_ok=True)
os.makedirs(PDF_DIR, exist_ok=True)

class TimeoutException(Exception):
    pass

@contextmanager
def time_limit(seconds):
    """Context manager to limit execution time."""
    def signal_handler(signum, frame):
        raise TimeoutException("Timed out!")
    
    # Set the signal handler
    signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)  # Disable the alarm

def get_paper_hashes():
    """Scrape all paper hashes from the main conference track."""
    print("Fetching paper list from NeurIPS 2024 proceedings...")
    
    response = requests.get(PAPERS_URL)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find all paper links with "Abstract-Conference.html" (main conference track)
    paper_links = soup.find_all('a', href=re.compile(r'/paper_files/paper/2024/hash/[a-f0-9]+-Abstract-Conference\.html'))
    
    hashes = []
    for link in paper_links:
        href = link.get('href')
        # Extract hash from URL: /paper_files/paper/2024/hash/{HASH}-Abstract-Conference.html
        match = re.search(r'/hash/([a-f0-9]+)-Abstract-Conference\.html', href)
        if match:
            hashes.append(match.group(1))
    
    print(f"Found {len(hashes)} papers in the main conference track")
    return hashes

def download_pdf(paper_hash, output_path):
    """Download PDF for a given paper hash."""
    pdf_url = f"{BASE_URL}/paper_files/paper/2024/file/{paper_hash}-Paper-Conference.pdf"
    
    try:
        response = requests.get(pdf_url, timeout=30)
        response.raise_for_status()
        
        with open(output_path, 'wb') as f:
            f.write(response.content)
        return True
    except Exception as e:
        print(f"Error downloading {paper_hash}: {e}")
        return False

def pdf_to_markdown(pdf_path, markdown_path, timeout=PDF_TIMEOUT):
    """Convert PDF to markdown using pymupdf4llm with timeout."""
    try:
        with time_limit(timeout):
            # Convert PDF to markdown
            md_text = pymupdf4llm.to_markdown(pdf_path)
            
            # Save markdown
            with open(markdown_path, 'w', encoding='utf-8') as f:
                f.write(md_text)
        return True, None
    except TimeoutException:
        return False, "timeout"
    except Exception as e:
        return False, f"error: {str(e)}"

def add_to_skipped(paper_hash, reason):
    """Add a paper to the skipped list with reason."""
    with open(SKIPPED_FILE, 'a') as f:
        f.write(f"{paper_hash}\t{reason}\t{time.strftime('%Y-%m-%d %H:%M:%S')}\n")

def get_existing_progress():
    """Check which papers have already been processed."""
    existing_markdowns = set()
    existing_pdfs = set()
    skipped_papers = set()
    
    if os.path.exists(MARKDOWN_DIR):
        existing_markdowns = {f.replace('.md', '') for f in os.listdir(MARKDOWN_DIR) if f.endswith('.md')}
    
    if os.path.exists(PDF_DIR):
        existing_pdfs = {f.replace('.pdf', '') for f in os.listdir(PDF_DIR) if f.endswith('.pdf')}
    
    if os.path.exists(SKIPPED_FILE):
        with open(SKIPPED_FILE, 'r') as f:
            skipped_papers = {line.strip().split('\t')[0] for line in f if line.strip()}
    
    return existing_markdowns, existing_pdfs, skipped_papers

def main():
    """Main function to download and convert all papers."""
    start_time = time.time()
    
    # Get all paper hashes
    paper_hashes = get_paper_hashes()
    
    if not paper_hashes:
        print("No papers found. Exiting.")
        return
    
    # Check existing progress
    existing_markdowns, existing_pdfs, skipped_papers = get_existing_progress()
    
    # Calculate what needs to be done (exclude already converted and skipped)
    papers_to_process = [h for h in paper_hashes if h not in existing_markdowns and h not in skipped_papers]
    papers_already_done = len(paper_hashes) - len(papers_to_process) - len(skipped_papers)
    
    print(f"\n{'='*60}")
    print(f"Progress Summary:")
    print(f"{'='*60}")
    print(f"Total papers in conference: {len(paper_hashes)}")
    print(f"Already converted to markdown: {papers_already_done}")
    print(f"Previously skipped (long/corrupted): {len(skipped_papers)}")
    print(f"Remaining to process: {len(papers_to_process)}")
    print(f"PDFs already downloaded: {len(existing_pdfs)}")
    print(f"{'='*60}\n")
    
    if not papers_to_process:
        print("All papers have been processed or skipped!")
        return
    
    # Process each paper
    successful_downloads = 0
    successful_conversions = 0
    failed_papers = []
    skipped_in_run = []
    processing_times = []
    
    print(f"Processing {len(papers_to_process)} remaining papers...")
    print(f"PDF conversion timeout: {PDF_TIMEOUT} seconds ({PDF_TIMEOUT/60:.1f} minutes)\n")
    
    for i, paper_hash in enumerate(tqdm(papers_to_process, desc="Processing papers")):
        paper_start_time = time.time()
        
        pdf_path = os.path.join(PDF_DIR, f"{paper_hash}.pdf")
        markdown_path = os.path.join(MARKDOWN_DIR, f"{paper_hash}.md")
        
        # Download PDF if not already downloaded
        if not os.path.exists(pdf_path):
            if download_pdf(paper_hash, pdf_path):
                successful_downloads += 1
            else:
                failed_papers.append(paper_hash)
                add_to_skipped(paper_hash, "download_failed")
                continue
        
        # Convert PDF to markdown with timeout
        success, error_reason = pdf_to_markdown(pdf_path, markdown_path)
        
        if success:
            successful_conversions += 1
            
            # Track processing time
            paper_elapsed = time.time() - paper_start_time
            processing_times.append(paper_elapsed)
            
            # Update ETA every 10 papers
            if (i + 1) % 10 == 0 and processing_times:
                avg_time = sum(processing_times) / len(processing_times)
                remaining = len(papers_to_process) - (i + 1)
                eta_seconds = avg_time * remaining
                eta_minutes = eta_seconds / 60
                eta_hours = eta_minutes / 60
                
                tqdm.write(f"Average time per paper: {avg_time:.2f}s | ETA: ", end="")
                if eta_hours >= 1:
                    tqdm.write(f"{eta_hours:.1f}h ({eta_minutes:.0f}m)")
                else:
                    tqdm.write(f"{eta_minutes:.1f}m ({eta_seconds:.0f}s)")
            
            os.remove(pdf_path)
        else:
            # Handle timeout or error
            if error_reason == "timeout":
                tqdm.write(f"⏱️  Timeout on {paper_hash} - skipping (too long)")
                add_to_skipped(paper_hash, "timeout_too_long")
                skipped_in_run.append((paper_hash, "timeout"))
            else:
                tqdm.write(f"❌ Error on {paper_hash}: {error_reason}")
                add_to_skipped(paper_hash, f"conversion_error")
                skipped_in_run.append((paper_hash, "error"))
            failed_papers.append(paper_hash)
    
    # Calculate elapsed time
    elapsed_time = time.time() - start_time
    elapsed_minutes = elapsed_time / 60
    elapsed_hours = elapsed_minutes / 60
    
    # Calculate average processing time
    avg_processing_time = sum(processing_times) / len(processing_times) if processing_times else 0
    
    # Summary
    print("\n" + "="*60)
    print("Processing Complete!")
    print("="*60)
    print(f"Total papers in conference: {len(paper_hashes)}")
    print(f"Already completed before this run: {papers_already_done}")
    print(f"Processed in this run: {successful_conversions}")
    print(f"Skipped in this run (timeout/error): {len(skipped_in_run)}")
    print(f"Total completed: {papers_already_done + successful_conversions}")
    print(f"Total skipped ever: {len(skipped_papers) + len(skipped_in_run)}")
    print(f"PDFs downloaded in this run: {successful_downloads}")
    print(f"\nPerformance Metrics:")
    print(f"  Average time per paper: {avg_processing_time:.2f} seconds")
    
    if elapsed_hours >= 1:
        print(f"  Total time taken: {elapsed_hours:.2f} hours ({elapsed_minutes:.1f} minutes)")
    else:
        print(f"  Total time taken: {elapsed_minutes:.1f} minutes ({elapsed_time:.1f} seconds)")
    
    if successful_conversions > 0:
        throughput = successful_conversions / elapsed_time * 60  # papers per minute
        print(f"  Throughput: {throughput:.2f} papers/minute")
    
    print(f"\nOutput directories:")
    print(f"  - PDFs: {PDF_DIR}")
    print(f"  - Markdown: {MARKDOWN_DIR}")
    
    if skipped_in_run:
        print(f"\nSkipped in this run ({len(skipped_in_run)}):")
        timeout_count = sum(1 for _, reason in skipped_in_run if reason == "timeout")
        error_count = len(skipped_in_run) - timeout_count
        print(f"  - Timeouts (too long): {timeout_count}")
        print(f"  - Conversion errors: {error_count}")
        print(f"  - Details saved to: {SKIPPED_FILE}")
    
    if len(skipped_papers) > 0:
        print(f"\nTotal skipped papers across all runs: {len(skipped_papers) + len(skipped_in_run)}")
        print(f"  See {SKIPPED_FILE} for details")
    
    print("="*60)

if __name__ == "__main__":
    main()

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Fetching paper list from NeurIPS 2024 proceedings...
Found 4035 papers in the main conference track

Progress Summary:
Total papers in conference: 4035
Already converted to markdown: 3740
Previously skipped (long/corrupted): 292
Remaining to process: 3
PDFs already downloaded: 0

Processing 3 remaining papers...
PDF conversion timeout: 180 seconds (3.0 minutes)



Processing papers: 100%|██████████| 3/3 [01:01<00:00, 20.40s/it]


Processing Complete!
Total papers in conference: 4035
Already completed before this run: 3740
Processed in this run: 3
Skipped in this run (timeout/error): 0
Total completed: 3743
Total skipped ever: 292
PDFs downloaded in this run: 3

Performance Metrics:
  Average time per paper: 20.34 seconds
  Total time taken: 1.1 minutes (66.2 seconds)
  Throughput: 2.72 papers/minute

Output directories:
  - PDFs: neurips_2024_papers/pdfs
  - Markdown: neurips_2024_papers/markdown

Total skipped papers across all runs: 292
  See neurips_2024_papers/skipped_papers.txt for details
